# 04 — One-zone network evolution


This notebook evolves the toy network in one zone. The solver API is designed to replace common NucNet Tools one-zone evolution workflows.

Available methods include fixed-step `euler`, `rk4`, and SciPy methods such as `bdf`, `radau`, and `lsoda` when SciPy is installed.

In [ ]:
from pathlib import Path
import sys

# Make the local editable package importable when the notebooks are run from the repo.
HERE = Path.cwd()
CANDIDATES = [HERE / "src", HERE.parent / "src", HERE.parent.parent / "src"]
for p in CANDIDATES:
    if (p / "nucnetpy").exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nucnetpy as nn
print("nucnetpy version:", nn.__version__)
def build_toy_alpha_network():
    """Build a small alpha-chain toy network for tutorials.

    This is not intended to be a physical helium-burning network.  The rates are
    deliberately simple constants so that the examples run quickly and are easy
    to inspect.
    """
    net = nn.Network()
    for name in ["he4", "be8", "c12"]:
        net.add_species(nn.Species.parse(name))

    r1 = nn.Reaction.from_names(
        ["he4", "he4"], ["be8"],
        constant_rate=2.0e-6,
        q_value=0.092,
        label="toy_2alpha_to_be8",
    )
    r2 = nn.Reaction.from_names(
        ["be8", "he4"], ["c12"],
        constant_rate=5.0e-5,
        q_value=7.367,
        label="toy_be8_alpha_to_c12",
    )
    net.reactions.add(r1)
    net.reactions.add(r2)

    zone = nn.Zone(label=("toy", "0", "0"), properties={"t9": "1.0", "rho": "1e4"})
    zone.set_abundance("he4", 0.25)  # X(he4) = A*Y = 1.0
    zone.set_abundance("be8", 0.0)
    zone.set_abundance("c12", 0.0)
    net.add_zone(zone)
    return net
net = build_toy_alpha_network()
zone = net.zone(0)

## Evolve with constant thermodynamics

In [ ]:
times = nn.time_grid(0.0, 2.0e-2, 200)
result = nn.evolve_zone(
    net, zone, times,
    thermo=nn.constant_thermo(t9=1.0, rho=1e4),
    method='bdf',
    rtol=1e-8,
    atol=1e-30,
)
print(result.success, result.message)
print(result.final_abundances)

In [ ]:
for i, name in enumerate(result.species):
    plt.plot(result.time, result.y[:, i], label=name)
plt.xlabel('time')
plt.ylabel('Y')
plt.yscale('log')
plt.legend();

## Convert to mass fractions

In [ ]:
X = result.mass_fraction_history(net.species)
for i, name in enumerate(result.species):
    plt.plot(result.time, X[:, i], label=name)
plt.xlabel('time')
plt.ylabel('X')
plt.yscale('log')
plt.legend();

## Compare fixed-step RK4 and BDF

For stiff networks, BDF/Radau are usually preferred. For small non-stiff tutorial networks, RK4 is useful for transparent tests.

In [ ]:
rk = nn.evolve_zone(net, zone, times, nn.constant_thermo(1.0, 1e4), method='rk4')
compare = pd.DataFrame({
    'species': result.species,
    'BDF_final_Y': [result.final_abundances[s] for s in result.species],
    'RK4_final_Y': [rk.final_abundances[s] for s in result.species],
})
compare['difference'] = compare['BDF_final_Y'] - compare['RK4_final_Y']
compare